# Detectra AI — Model Export & INT8 Quantization
Run this after training to quantize all custom models for CPU deployment.
**Output:** logo_vit_quantized.onnx, motion_videomae_quantized.onnx


In [ ]:
!pip install -q onnx onnxruntime onnxruntime-tools
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType
import numpy as np, time, os
from pathlib import Path
print("ORT version:", ort.__version__)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
MODEL_DIR = Path("/content/drive/MyDrive/detectra_models")
print("Models in Drive:", list(MODEL_DIR.glob("*.onnx")))

In [ ]:
def quantize_model(src, dst, weight_type=QuantType.QInt8):
    quantize_dynamic(str(src), str(dst), weight_type=weight_type)
    orig = src.stat().st_size/1e6
    quant = dst.stat().st_size/1e6
    print(f"{src.name}: {orig:.1f}MB -> {dst.name}: {quant:.1f}MB ({(1-quant/orig)*100:.0f}% smaller)")

def benchmark(model_path, input_shape, name, n=20):
    sess = ort.InferenceSession(str(model_path), providers=["CPUExecutionProvider"])
    inp_name = sess.get_inputs()[0].name
    dummy = np.random.rand(*input_shape).astype(np.float32)
    for _ in range(3): sess.run(None, {inp_name: dummy})  # warmup
    t0 = time.time()
    for _ in range(n): sess.run(None, {inp_name: dummy})
    ms = (time.time()-t0)/n*1000
    print(f"{name}: {ms:.1f}ms/inference ({1000/ms:.0f} FPS)")

In [ ]:
# Quantize Logo ViT
logo_src = MODEL_DIR / "logo_vit.onnx"
logo_dst = MODEL_DIR / "logo_vit_quantized.onnx"
if logo_src.exists():
    quantize_model(logo_src, logo_dst)
    benchmark(logo_dst, (1,3,224,224), "Logo ViT (quantized)")
else:
    print("logo_vit.onnx not found - run notebook 01 first")

In [ ]:
# Quantize Motion VideoMAE
motion_src = MODEL_DIR / "motion_videomae.onnx"
motion_dst = MODEL_DIR / "motion_videomae_quantized.onnx"
if motion_src.exists():
    quantize_model(motion_src, motion_dst)
    benchmark(motion_dst, (1,16,3,224,224), "Motion VideoMAE (quantized)")
else:
    print("motion_videomae.onnx not found - run notebook 02 first")

In [ ]:
# Summary
print("
=== Models ready for deployment ===")
for f in MODEL_DIR.glob("*.onnx"):
    print(f"  {f.name}: {f.stat().st_size/1e6:.1f} MB")
for f in MODEL_DIR.glob("*.pt"):
    print(f"  {f.name}: {f.stat().st_size/1e6:.1f} MB")
print("
Copy all files to: backend/models/")